In [1]:
import pandas as pd
import numpy as np

def load_clinical_data(filepath):
    """
    Architectural Logic: Safe Loading.
    """
    try:
        df = pd.read_csv(filepath)
        print(f"✅ Success: Loaded data from {filepath}")
        return df
    except FileNotFoundError:
        print(f"❌ Error: Could not find the file at {filepath}")
        return None

# Load the data 
df = load_clinical_data('diabetes.csv')

✅ Success: Loaded data from diabetes.csv


In [4]:
# 1. Technical Audit: How many rows/columns? Are types correct?
print("--- TECHNICAL AUDIT (info) ---")
df.info()

# 2. Statistical Audit: Look at Min/Max/Mean
print("\n--- STATISTICAL AUDIT (describe) ---")
df.describe()

--- TECHNICAL AUDIT (info) ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               768 non-null    int64  
 1   Glucose                   768 non-null    int64  
 2   BloodPressure             768 non-null    int64  
 3   SkinThickness             768 non-null    int64  
 4   Insulin                   768 non-null    int64  
 5   BMI                       768 non-null    float64
 6   DiabetesPedigreeFunction  768 non-null    float64
 7   Age                       768 non-null    int64  
 8   Outcome                   768 non-null    int64  
dtypes: float64(2), int64(7)
memory usage: 54.1 KB

--- STATISTICAL AUDIT (describe) ---


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
count,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000
mean,3.845052,120.894531,69.105469,20.536458,79.799479,31.992578,0.471876,33.240885,0.348958
std,3.369578,31.972618,19.355807,15.952218,115.244002,7.884160,0.331329,11.760232,0.476951
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.078000,21.000000,0.000000
25%,1.000000,99.000000,62.000000,0.000000,0.000000,27.300000,0.243750,24.000000,0.000000
50%,3.000000,117.000000,72.000000,23.000000,30.500000,32.000000,0.372500,29.000000,0.000000
75%,6.000000,140.250000,80.000000,32.000000,127.250000,36.600000,0.626250,41.000000,1.000000
max,17.000000,199.000000,122.000000,99.000000,846.000000,67.100000,2.420000,81.000000,1.000000


In [5]:
# 1. Select only the columns where '0' is medically impossible
# Architect's Logic: 'Pregnancies' can be 0. 'Glucose' cannot.
cols_to_sanitize = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']

def sanitize_false_zeros(df, list_of_cols):
    """
    Logic: This tool finds 0 and labels it as 'No Data' (NaN).
    """
    # Replace 0 with NaN (Not a Number)
    # NaN is the language of 'Missing Truth' in Python/NumPy
    df[list_of_cols] = df[list_of_cols].replace(0, np.nan)
    
    # Audit: How many gaps did we reveal?
    null_counts = df[list_of_cols].isnull().sum()
    print("\n🔍 Clinical Data Audit (Hidden Truth Revealed):")
    print(null_counts)
    
    return df

# Run the sanitization
df = sanitize_false_zeros(df, cols_to_sanitize)


🔍 Clinical Data Audit (Hidden Truth Revealed):
Glucose            5
BloodPressure     35
SkinThickness    227
Insulin          374
BMI               11
dtype: int64


In [8]:
from sklearn.impute import KNNImputer

# 1. Define the 'Target Columns' (What the computer didn't know before)
cols_to_fill = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']

# 2. Logic: Setup the Smart Imputer (Looking at 5 neighbors)
imputer = KNNImputer(n_neighbors=5)

def clinical_knn_imputation(df, list_of_cols):
    """
    Architecture: Fills gaps by comparing with similar patient profiles.
    """
    # Use the imputer on the target columns
    df[list_of_cols] = imputer.fit_transform(df[list_of_cols])
    
    print("✅ Success: KNN Imputation complete for clinical pillars.")
    return df

# 3. Run the complete procedure
df = clinical_knn_imputation(df, cols_to_fill)

# Final Audit: To prove the gaps are gone
print("\n🔍 Final Integrity Audit (Nulls remaining):")
print(df.isnull().sum())


✅ Success: KNN Imputation complete for clinical pillars.

🔍 Final Integrity Audit (Nulls remaining):
Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64
